# AresSim training evaluation report

This checkout template is executed by `aresim-rl report`. It is not shipped inside the `aresim` package. `RUN_DIRECTORY` is injected before execution.

> **Interpretation warning:** `phase1_open_exploration_v1` has no success condition. Returns, survival, exploration, legality, safety, and resource behavior are diagnostics; this report does not promote a scientifically best checkpoint.

In [2]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
import wandb
run = Path(r"../results/rllib_masked_ppo_smoke/seed_7")
manifest = json.loads((run / 'manifest.json').read_text())
tracking = manifest['experiment']['tracking']
api = wandb.Api()
entity = tracking['entity'] or api.default_entity
tracked_run = api.run(f"{entity}/{tracking['project']}/{manifest['wandb_run_id']}")
metrics = pd.DataFrame(tracked_run.scan_history())
if '_step' in metrics: metrics['step'] = metrics['_step']
manifest['status'], manifest['config_hash'], len(metrics)

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /Users/shanmukh/.netrc.


CommError: Could not find run <Run shanmukh/aresim/552214b34e1d64c886677528 (not found)>

## Training curves

In [ ]:
reward_terms = [name for name in metrics if name.startswith('train/reward_term/')]
action_metrics = [name for name in metrics if name.startswith('train/action_count/')]
telemetry = [name for name in metrics if name.startswith('train/telemetry/')]
groups = [
    ['train/shaped_return', 'train/episode_length'],
    ['learner/total_loss', 'learner/policy_loss', 'learner/value_loss'],
    ['learner/entropy', 'learner/approx_kl', 'learner/clip_fraction'],
    reward_terms,
    action_metrics,
    telemetry + ['system/environment_steps_per_second'],
]
x_axis = 'train/environment_steps' if 'train/environment_steps' in metrics else 'step'
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, names in zip(axes.flat, groups):
    for name in names:
        if name in metrics: ax.plot(metrics[x_axis], metrics[name], label=name)
    ax.set_xlabel('environment steps'); ax.grid(alpha=.25); ax.legend(fontsize=8)
fig.tight_layout(); fig

## Frozen evaluation and trajectory artifacts

In [ ]:
summary_files = sorted((run / 'evaluation').glob('*/summary.json')) if (run / 'evaluation').exists() else []
summaries = [json.loads(path.read_text()) for path in summary_files]
baseline_rows = [row for summary in summaries for row in summary.get('baseline_comparisons', [])]
pd.DataFrame(summaries), pd.DataFrame(baseline_rows), [str(path) for path in sorted((run / 'evaluation').glob('*/trajectories/episodes/*.json'))[:5]]